In [ ]:
from google.colab import files
files = files.upload()

Saving blinkit_inventoryNew.csv to blinkit_inventoryNew.csv
Saving blinkit_marketing_performance.csv to blinkit_marketing_performance.csv
Saving blinkit_order_items.csv to blinkit_order_items.csv
Saving blinkit_orders.csv to blinkit_orders.csv
Saving blinkit_products.csv to blinkit_products.csv
Saving blinkit_inventory.csv to blinkit_inventory.csv
Saving blinkit_customer_feedback.csv to blinkit_customer_feedback.csv
Saving blinkit_customers.csv to blinkit_customers.csv
Saving blinkit_delivery_performance.csv to blinkit_delivery_performance.csv
Saving Category_Icons.xlsx to Category_Icons.xlsx
Saving Rating_Icon.xlsx to Rating_Icon.xlsx


In [ ]:
import os
os.listdir("/content")

['.config',
 'blinkit_inventoryNew.csv',
 'Rating_Icon.xlsx',
 'blinkit_order_items.csv',
 'blinkit_marketing_performance.csv',
 'blinkit_inventory.csv',
 'blinkit_delivery_performance.csv',
 'Category_Icons.xlsx',
 'blinkit_customer_feedback.csv',
 'blinkit_customers.csv',
 'blinkit_orders.csv',
 'blinkit_products.csv',
 'sample_data']

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
import zipfile
from google.colab import files

In [ ]:
orders = pd.read_csv("/content/blinkit_orders.csv")
order_items = pd.read_csv("/content/blinkit_order_items.csv")
customers = pd.read_csv("/content/blinkit_customers.csv")
products = pd.read_csv("/content/blinkit_products.csv")
delivery = pd.read_csv("/content/blinkit_delivery_performance.csv")
feedback = pd.read_csv("/content/blinkit_customer_feedback.csv")
inventory = pd.read_csv("/content/blinkit_inventory.csv")
inventory_monthly = pd.read_csv("/content/blinkit_inventoryNew.csv")
marketing = pd.read_csv("/content/blinkit_marketing_performance.csv")

category_icons = pd.read_excel("/content/Category_Icons.xlsx")
rating_icons = pd.read_excel("/content/Rating_Icon.xlsx")

In [ ]:
tables = {
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "products": products,
    "delivery": delivery,
    "feedback": feedback,
    "inventory": inventory,
    "inventory_monthly": inventory_monthly,
    "marketing": marketing
}

for name, df in tables.items():
    print("\n==============================")
    print(name)
    print("Rows, Columns:", df.shape)
    print(df.head())


orders
Rows, Columns: (5000, 10)
     order_id  customer_id           order_date promised_delivery_time  \
0  1961864118     30065862  2024-07-17 08:34:01    2024-07-17 08:52:01   
1  1549769649      9573071  2024-05-28 13:14:29    2024-05-28 13:25:29   
2  9185164487     45477575  2024-09-23 13:07:12    2024-09-23 13:25:12   
3  9644738826     88067569  2023-11-24 16:16:56    2023-11-24 16:34:56   
4  5427684290     83298567  2023-11-20 05:00:39    2023-11-20 05:17:39   

  actual_delivery_time delivery_status  order_total payment_method  \
0  2024-07-17 08:47:01         On Time      3197.07           Cash   
1  2024-05-28 13:27:29         On Time       976.55           Cash   
2  2024-09-23 13:29:12         On Time       839.05            UPI   
3  2023-11-24 16:33:56         On Time       440.23           Card   
4  2023-11-20 05:18:39         On Time      2526.68           Cash   

   delivery_partner_id  store_id  
0                63230      4771  
1                14983      75

In [ ]:
for name, df in tables.items():
    print("\n", name)
    print(df.isnull().sum())


 orders
order_id                  0
customer_id               0
order_date                0
promised_delivery_time    0
actual_delivery_time      0
delivery_status           0
order_total               0
payment_method            0
delivery_partner_id       0
store_id                  0
dtype: int64

 order_items
order_id      0
product_id    0
quantity      0
unit_price    0
dtype: int64

 customers
customer_id          0
customer_name        0
email                0
phone                0
address              0
area                 0
pincode              0
registration_date    0
customer_segment     0
total_orders         0
avg_order_value      0
dtype: int64

 products
product_id           0
product_name         0
category             0
brand                0
price                0
mrp                  0
margin_percentage    0
shelf_life_days      0
min_stock_level      0
max_stock_level      0
dtype: int64

 delivery
order_id                    0
delivery_partner_id         0
prom

In [ ]:
for name, df in tables.items():
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

In [ ]:
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["promised_delivery_time"] = pd.to_datetime(orders["promised_delivery_time"])
orders["actual_delivery_time"] = pd.to_datetime(orders["actual_delivery_time"])

delivery["promised_time"] = pd.to_datetime(delivery["promised_time"])
delivery["actual_time"] = pd.to_datetime(delivery["actual_time"])

customers["registration_date"] = pd.to_datetime(customers["registration_date"])
feedback["feedback_date"] = pd.to_datetime(feedback["feedback_date"])

inventory["date"] = pd.to_datetime(inventory["date"], dayfirst=True)

inventory_monthly["date"] = pd.to_datetime(
    inventory_monthly["date"],
    format="%b-%y"
)

marketing["date"] = pd.to_datetime(marketing["date"])

In [ ]:
delivery["reasons_if_delayed"] = delivery["reasons_if_delayed"].fillna("No Delay")

In [ ]:
orders = orders.drop_duplicates()
order_items = order_items.drop_duplicates()
customers = customers.drop_duplicates()
products = products.drop_duplicates()
delivery = delivery.drop_duplicates()
feedback = feedback.drop_duplicates()
inventory = inventory.drop_duplicates()
inventory_monthly = inventory_monthly.drop_duplicates()
marketing = marketing.drop_duplicates()

In [ ]:
sales_data = (
    orders
    .merge(order_items, on="order_id", how="left")
    .merge(customers, on="customer_id", how="left")
    .merge(products, on="product_id", how="left")
    .merge(
        delivery[[
            "order_id",
            "delivery_time_minutes",
            "distance_km",
            "delivery_status",
            "reasons_if_delayed"
        ]],
        on="order_id",
        how="left"
    )
    .merge(
        feedback[[
            "order_id",
            "rating",
            "feedback_category",
            "sentiment"
        ]],
        on="order_id",
        how="left"
    )
)

In [ ]:
sales_data["item_revenue"] = sales_data["quantity"] * sales_data["unit_price"]

sales_data["profit_amount"] = (
    sales_data["item_revenue"] * sales_data["margin_percentage"] / 100
)

sales_data["order_month"] = sales_data["order_date"].dt.to_period("M").astype(str)
sales_data["order_day"] = sales_data["order_date"].dt.day_name()
sales_data["order_hour"] = sales_data["order_date"].dt.hour

In [ ]:
sales_data.head()
sales_data.shape
sales_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   order_id                5000 non-null   int64         
 1   customer_id             5000 non-null   int64         
 2   order_date              5000 non-null   datetime64[ns]
 3   promised_delivery_time  5000 non-null   datetime64[ns]
 4   actual_delivery_time    5000 non-null   datetime64[ns]
 5   delivery_status_x       5000 non-null   object        
 6   order_total             5000 non-null   float64       
 7   payment_method          5000 non-null   object        
 8   delivery_partner_id     5000 non-null   int64         
 9   store_id                5000 non-null   int64         
 10  product_id              5000 non-null   int64         
 11  quantity                5000 non-null   int64         
 12  unit_price              5000 non-null   float64 

In [ ]:
orders.to_csv("orders_cleaned.csv", index=False)
order_items.to_csv("order_items_cleaned.csv", index=False)
customers.to_csv("customers_cleaned.csv", index=False)
products.to_csv("products_cleaned.csv", index=False)
delivery.to_csv("delivery_cleaned.csv", index=False)
feedback.to_csv("feedback_cleaned.csv", index=False)
inventory.to_csv("inventory_cleaned.csv", index=False)
inventory_monthly.to_csv("inventory_monthly_cleaned.csv", index=False)
marketing.to_csv("marketing_cleaned.csv", index=False)
sales_data.to_csv("sales_data_cleaned.csv", index=False)

In [ ]:
cleaned_files = [
    "orders_cleaned.csv",
    "order_items_cleaned.csv",
    "customers_cleaned.csv",
    "products_cleaned.csv",
    "delivery_cleaned.csv",
    "feedback_cleaned.csv",
    "inventory_cleaned.csv",
    "inventory_monthly_cleaned.csv",
    "marketing_cleaned.csv",
    "sales_data_cleaned.csv"
]

with zipfile.ZipFile("blinkit_cleaned_data.zip", "w") as zipf:
    for file in cleaned_files:
        zipf.write(file)

In [ ]:
files.download("blinkit_cleaned_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
conn = sqlite3.connect("blinkit.db")

orders.to_sql("orders", conn, if_exists="replace", index=False)
order_items.to_sql("order_items", conn, if_exists="replace", index=False)
customers.to_sql("customers", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
delivery.to_sql("delivery", conn, if_exists="replace", index=False)
feedback.to_sql("feedback", conn, if_exists="replace", index=False)
inventory.to_sql("inventory", conn, if_exists="replace", index=False)
inventory_monthly.to_sql("inventory_monthly", conn, if_exists="replace", index=False)
marketing.to_sql("marketing", conn, if_exists="replace", index=False)
sales_data.to_sql("sales_data", conn, if_exists="replace", index=False)

5000

In [ ]:
pd.read_sql("""
SELECT
    ROUND(SUM(item_revenue), 2) AS total_revenue
FROM sales_data;
""", conn)

,total_revenue
0,4972415.43


In [ ]:
revenue_by_category = pd.read_sql("""
SELECT
    category,
    ROUND(SUM(item_revenue), 2) AS revenue,
    SUM(quantity) AS total_quantity
FROM sales_data
GROUP BY category
ORDER BY revenue DESC;
""", conn)

revenue_by_category

,category,revenue,total_quantity
0,Dairy & Breakfast,639222.19,1114
1,Pharmacy,592368.57,973
2,Fruits & Vegetables,559053.08,966
3,Pet Care,539888.75,1003
4,Household Care,444244.25,1078
5,Personal Care,394894.61,887
6,Snacks & Munchies,394648.71,963
7,Cold Drinks & Juices,392717.62,758
8,Grocery & Staples,359937.82,895
9,Baby Care,348227.18,655


In [ ]:
top_products = pd.read_sql("""
SELECT
    product_name,
    category,
    ROUND(SUM(item_revenue), 2) AS revenue,
    SUM(quantity) AS quantity_sold
FROM sales_data
GROUP BY product_name, category
ORDER BY revenue DESC
LIMIT 10;
""", conn)

top_products

,product_name,category,revenue,quantity_sold
0,Vitamins,Pharmacy,260822.01,380
1,Pet Treats,Pet Care,252007.37,473
2,Cough Syrup,Pharmacy,203569.98,373
3,Toilet Cleaner,Household Care,199837.48,430
4,Bread,Dairy & Breakfast,184851.10,270
5,Dish Soap,Household Care,184441.21,397
6,Cat Food,Pet Care,166596.39,307
7,Baby Wipes,Baby Care,158768.41,328
8,Onions,Fruits & Vegetables,138858.42,222
9,Baby Food,Baby Care,137442.79,236


In [ ]:
delivery_status_summary = pd.read_sql("""
SELECT
    delivery_status,
    COUNT(*) AS total_orders,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS percentage
FROM orders
GROUP BY delivery_status;
""", conn)

delivery_status_summary

,delivery_status,total_orders,percentage
0,On Time,3470,69.40
1,Significantly Delayed,493,9.86
2,Slightly Delayed,1037,20.74


In [ ]:
sentiment_summary = pd.read_sql("""
SELECT
    sentiment,
    COUNT(*) AS total_feedback,
    ROUND(AVG(rating), 2) AS avg_rating
FROM feedback
GROUP BY sentiment;
""", conn)

sentiment_summary

,sentiment,total_feedback,avg_rating
0,Negative,1642,2.01
1,Neutral,1738,3.52
2,Positive,1620,4.50


In [ ]:
marketing_roas = pd.read_sql("""
SELECT
    channel,
    ROUND(SUM(spend), 2) AS total_spend,
    ROUND(SUM(revenue_generated), 2) AS revenue_generated,
    ROUND(SUM(revenue_generated) / SUM(spend), 2) AS roas
FROM marketing
GROUP BY channel
ORDER BY roas DESC;
""", conn)

marketing_roas

,channel,total_spend,revenue_generated,roas
0,Email,3997488.04,8189331.58,2.05
1,SMS,3998607.54,7938649.32,1.99
2,Social Media,4110363.91,7990415.98,1.94
3,App,4213378.75,8075010.49,1.92


In [ ]:
revenue_by_category.to_csv("sql_revenue_by_category.csv", index=False)
top_products.to_csv("sql_top_products.csv", index=False)
delivery_status_summary.to_csv("sql_delivery_status_summary.csv", index=False)
sentiment_summary.to_csv("sql_sentiment_summary.csv", index=False)
marketing_roas.to_csv("sql_marketing_roas.csv", index=False)

In [ ]:
sql_files = [
    "sql_revenue_by_category.csv",
    "sql_top_products.csv",
    "sql_delivery_status_summary.csv",
    "sql_sentiment_summary.csv",
    "sql_marketing_roas.csv"
]

with zipfile.ZipFile("blinkit_sql_outputs.zip", "w") as zipf:
    for file in sql_files:
        zipf.write(file)

files.download("blinkit_sql_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>